# Notebook 1: Setup and Train Your First Model

**For beginners.** This notebook will:
1. Check if your GPU is working
2. Install what you need
3. Train a small model step by step

**Estimated time:** 30 minutes on A100 GPU

**What is happening?** We teach a small AI model (student) to copy a big AI model (teacher). This is called "Knowledge Distillation".


## Step 1: Check Your GPU

Before training, we need to make sure the GPU is visible. Run the cell below.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU found: {gpu_name}")
    print(f"✅ GPU memory: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU found! Training will be very slow.")
    print("   Please check your cloud GPU settings.")

**What you should see:** A GPU name (like "NVIDIA A100") and memory size (like 80 GB).

If you see "No GPU found", stop here and ask for help.

## Step 2: Set GPU Memory Limit (Safety)

We only use 80% of GPU memory. This prevents the GPU from crashing.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.env import set_gpu_memory_fraction

set_gpu_memory_fraction(0.8)
print("✅ GPU memory limited to 80%")

## Step 3: Set Your HuggingFace Token

Some models need a login token. Get yours free at: https://huggingface.co/settings/tokens

Replace `YOUR_TOKEN_HERE` with your real token.

In [ ]:
import os

# Replace with your real token (starts with hf_)
os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'

print("✅ Token set (hidden for safety)")

## Step 4: Load Configuration

This reads the `configs/default.yaml` file. It tells the code:
- Which teacher model to use (the big model)
- Which student model to use (the small model we train)
- How many training steps
- Learning rate, batch size, etc.

In [ ]:
from src.config import load_config

config = load_config('../configs/default.yaml')

print("📋 Configuration loaded!")
print(f"   Teacher: {config.models['teacher'].name}")
print(f"   Student: {config.models['student'].name}")
print(f"   Epochs: {config.training.num_train_epochs}")
print(f"   Batch size: {config.training.per_device_train_batch_size}")
print(f"   Learning rate: {config.training.learning_rate}")

## Step 5: Load the Dataset

We use the "dolly" dataset. It has instructions and answers.

**What is a dataset?** Just a big table with text. The model learns from this text.

In [ ]:
from src.data.dataset_loader import load_and_prepare_dataset
from src.data.preprocessing import preprocess_dataset

print("📥 Downloading dataset...")
dataset = load_and_prepare_dataset(config.dataset, seed=42)

print("🔧 Preprocessing dataset...")
dataset = preprocess_dataset(dataset, config.dataset)

print(f"✅ Dataset ready!")
print(f"   Train examples: {len(dataset['train']):,}")
print(f"   Validation examples: {len(dataset['validation']):,}")

## Step 6: Load the Teacher Model (Big Model)

The teacher is a large model (7 billion parameters). It is already trained.

**Why load it?** The teacher gives "soft answers" that help the student learn better than just using correct/wrong labels.

⚠️ This step downloads ~15 GB. It takes 2-5 minutes.

In [ ]:
from src.models.teacher_loader import load_teacher_model

print("🧠 Loading teacher model...")
teacher, teacher_tokenizer = load_teacher_model(config.models['teacher'])

print("✅ Teacher loaded!")
print(f"   Model: {config.models['teacher'].name}")
print(f"   Device: {teacher.device}")

## Step 7: Load the Student Model (Small Model)

The student is a smaller model (1.5 billion parameters). This is the model we will train.

We also add "LoRA" — a trick that lets us train only a small part of the model, saving memory.

In [ ]:
from src.models.student_loader import load_student_model

print("🎓 Loading student model...")
student, student_tokenizer = load_student_model(
    config.models['student'],
    lora_config=config.lora
)

print("✅ Student loaded!")
print(f"   Model: {config.models['student'].name}")
print(f"   LoRA rank: {config.lora.r}")

## Step 8: Prepare Data for Training

We convert text into numbers (tokens) that the model understands.

We also create a "data collator" — it puts examples into batches.

In [ ]:
from src.data.tokenization import tokenize_dataset, get_tokenizer
from src.data.collators import DataCollatorForCausalLM

print("🔤 Tokenizing dataset...")
tokenized_dataset = tokenize_dataset(dataset, student_tokenizer, config.tokenization)

print("📦 Creating data collator...")
data_collator = DataCollatorForCausalLM(
    tokenizer=student_tokenizer,
    max_length=config.tokenization.max_length
)

print("✅ Data ready for training!")
print(f"   Max length: {config.tokenization.max_length} tokens")

## Step 9: Create the Trainer

The trainer is the "coach" that tells the student how to learn.

It uses:
- **Cross-Entropy Loss** — how wrong the student is
- **KL Divergence Loss** — how different the student is from the teacher

Together they make the student smart AND similar to the teacher.

In [ ]:
from src.models.distillation import create_distillation_trainer
import os

output_dir = '../artifacts/checkpoints'
os.makedirs(output_dir, exist_ok=True)

print("🏋️ Creating trainer...")
trainer = create_distillation_trainer(
    student_model=student,
    teacher_model=teacher,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=student_tokenizer,
    data_collator=data_collator,
    output_dir=output_dir,
    temperature=config.distillation.temperature,
    alpha=config.distillation.alpha,
    beta=config.distillation.beta,
    num_train_epochs=config.training.num_train_epochs,
    per_device_train_batch_size=config.training.per_device_train_batch_size,
    gradient_accumulation_steps=config.training.gradient_accumulation_steps,
    learning_rate=config.training.learning_rate,
    weight_decay=config.training.weight_decay,
    warmup_ratio=config.training.warmup_ratio,
    logging_steps=10,
    eval_steps=100,
    save_steps=50,
    bf16=config.hardware.mixed_precision == 'bf16',
    fp16=config.hardware.mixed_precision == 'fp16',
    dataloader_num_workers=config.training.dataloader_num_workers,
    gradient_checkpointing=config.hardware.gradient_checkpointing,
)

print("✅ Trainer ready!")

## Step 10: START TRAINING! 🚀

This is the main training loop. You will see output like:

```
Step 10 | Loss: 2.5 | LR: 0.0002
Step 20 | Loss: 2.3 | LR: 0.00018
```

**The Loss should go DOWN.** If it goes down, your model is learning! ✅

**Time:** About 30 minutes for 3 epochs.

In [ ]:
print("🚀 STARTING TRAINING!")
print("=" * 50)

trainer.train()

print("=" * 50)
print("🎉 TRAINING COMPLETE!")

## Step 11: Save Your Model

After training, we save the model so you can use it later.

In [ ]:
final_dir = '../artifacts/best_model/final'
os.makedirs(final_dir, exist_ok=True)

print("💾 Saving model...")
trainer.save_model(final_dir)
student_tokenizer.save_pretrained(final_dir)

print(f"✅ Model saved to: {final_dir}")
print("")
print("Next steps:")
print("  1. Run notebook 02 to find better settings")
print("  2. Run notebook 03 to fine-tune")
print("  3. Use notebook 04 to check progress anytime")

## ✅ You Did It!

You just trained your first distilled model!

**What to do next:**
- Open `02_optimize_round1.ipynb` to find better hyperparameters
- Open `04_check_progress.ipynb` to monitor training anytime